<div align="center">
  <img src="asset/Day2.png" alt="Databricks 14 Days AI Challenge - Day 02" width="800"/>
</div>

# ⚡ Day 2: Apache Spark Fundamentals
**14-Days of AI Challenge** | *Author: Tanmay Patil*

### 📝 Overview
Today we move from simple data loading to understanding the **Core Mechanics of Apache Spark**. We will explore how Spark processes data using Drivers, Executors, and the DAG (Directed Acyclic Graph), and why "Lazy Evaluation" makes it so efficient.

### 🎯 Learning Objectives
1.  **Spark Architecture:** Understand Driver vs. Executor roles.
2.  **DataFrames:** Working with structured data (Rows/Cols) vs RDDs.
3.  **Lazy Evaluation:** Why Spark waits to run code until you ask for results.
4.  **Magic Commands:** Using `%fs` and `%python` to navigate the environment.

## 🪄 Part 1: Notebook Magic Commands
Databricks notebooks allow us to mix languages and system commands using "Magic" (%).
* `%fs`: File System commands (like `ls` to list files).
* `%sh`: Shell commands.
* `%sql`: Run SQL queries directly on DataFrames.

In [0]:
# MAGIC %fs
# MAGIC # Let's list the files in our Volume to confirm the data is there
# MAGIC ls /Volumes/workspace/ecommerce/ecommerce_data/

## 🐢 Part 2: Lazy Evaluation in Action
In Spark, **transformations** (like `read`, `filter`, `select`) are **lazy**. When you run the cell below, Spark *does not actually read the file*. It simply creates a "plan" (DAG) to do so.

Processing only happens when we trigger an **action** (like `show`, `count`, or `write`).

In [0]:
# Define path to the November dataset
file_path = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv"

# TRANSFORMATION: This executes instantly because no data is moved yet.
# We are just telling the Driver *how* to load the data.
df_events = spark.read.option("header", "true") \
                      .option("inferSchema", "true") \
                      .csv(file_path)

print("✅ DataFrame Plan created (Lazy Evaluation). No data scanned yet.")

## 🛠️ Part 3: Select, Filter & Count
Now we perform **Actions**. When we run `.count()` or `.show()`, the Driver sends tasks to Executors to actually process the data.

In [0]:
from pyspark.sql.functions import col

# 1. SELECT: View specific columns
print("--- Sample Data (First 10 rows) ---")
df_events.select("event_type", "product_id", "price").show(10)

# 2. FILTER: Find expensive items (> $100)
# This is an ACTION that triggers a computation job.
expensive_count = df_events.filter(col("price") > 100).count()

print(f"💰 Number of transactions > $100: {expensive_count:,}")

## 📊 Part 4: Aggregations (The Power of Distributed Computing)
We will now compute the **Top 5 Brands** by activity. Spark performs this by "shuffling" data across the cluster so that all records for the same brand end up on the same executor node for counting.

In [0]:
# 1. Simple GroupBy: Count events by type (view, cart, purchase)
print("--- Event Distribution ---")
df_events.groupBy("event_type").count().show()

# 2. Complex Chain: Find Top 5 Brands
# Notice the clean chaining syntax used in production code
top_brands_df = (
    df_events
    .where(col("brand").isNotNull())  # Remove null brands
    .groupBy("brand")
    .count()
    .orderBy(col("count").desc())     # Sort Descending
    .limit(5)
)

print("--- Top 5 Popular Brands ---")
top_brands_df.show()

## 💾 Part 5: Saving Results
One of the most common tasks is saving your analysis. We will save our `top_brands_df` back to our Volume as a **CSV file** so it can be used for reporting later.

In [0]:
output_path = "/Volumes/workspace/ecommerce/ecommerce_data/top_brands_summary"

# Write the small summary dataframe to CSV
# mode("overwrite") replaces the file if it already exists
top_brands_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print(f"✅ Successfully exported Top Brands data to: {output_path}")

In [0]:
# Verify the file was created using Databricks utility
display(
    dbutils.fs.ls("/Volumes/workspace/ecommerce/ecommerce_data/top_brands_summary")
)